# 실험 G·ND: 함수 토큰 모델 vs 일반 함수 호출 모델 (논문에 없는 추가)
설계·예측·판정 규칙은 `EXPERIMENT_DESIGN.md` 4.5절 (돌리기 전에 적어 둠). 코드는 `src/general.py`, `src/train.py --format json [--no-list --json-desc]`.

**묻는 것**: 지금까지 본 거절 현상(낯선 말투 positive 과잉 거절, near-miss 오호출, 문턱값 이식 실패, 말투 의존)이
함수 토큰 모델이라서 생기나, 함수 호출 모델이면 공통으로 생기나.

일반 모델 G 는 ours 와 두 가지가 다르다: 정답 기호(새 토큰 vs 함수 이름), 설명을 쓰는 방식(정답 뒤에서 외우기 vs 입력에서 보고 고르기).
가운데에 ND(N+설명)를 두면 한 단계에 한 가지씩만 바뀐다.

```
ours ──────────────▶ ND ──────────────▶ G
     정답 기호만 바뀜          설명 위치만 바뀜
```

| | `ours` (기존, 다시 학습 안 함) | `ND` (새로) | `G` (새로) |
|---|---|---|---|
| 프롬프트 | 지시문 + 질의 | ours 와 똑같음 | 지시문 + **함수 21개 JSON 스키마** + 질의 (약 2,350 토큰) |
| 정답 | ` <nexa_11>(5)<nexa_end>` | ` {"name": "change_screen_brightness", "arguments": {"level": 5}}` | ND 와 같음 |
| 정답 뒤 | `Function description` + 설명 원문 | ours 와 똑같음 | 없음 (설명은 입력에 있다) |
| 거절 | `<nexa_20>` | `{"name": "irrelevant_function", "arguments": {}}` | 같음 (목록 21번째) |
| 학습 | (기존 A50) | `--format json --no-list --json-desc` | `--format json` |

| 비교 | 묻는 것 |
|---|---|
| **ours vs G (주 비교)** | 함수 토큰 모델만의 현상인가 |
| ours vs ND | 차이가 난다면 새 토큰이라서인가 |
| ND vs G | 아니면 설명을 입력에서 보고 고르기 때문인가 |

**같게 두는 것 (ours 학습 조건 그대로)**: gemma-2b full FT, `data/train.jsonl` 4,000개, AdamW(`adamw_torch`) lr 5e-5, warm-up 10, linear, weight decay 0,
3 epoch(750 step), 실제 batch 16, seed 0, A100 80GB. 평가도 같은 test·greedy·`max_new_tokens` 128·같은 채점기.
batch 16 을 나눠 담는 것(`--batch-size × --grad-accum`)과 gradient checkpointing 은 계산 결과가 같다.

**거절 점수**: ours 는 기존 `src.probs` (전체 사전 기준 확률). ND·G 는 함수 이름이 여러 토큰이라 이름 21개를 하나씩 정답 자리에 넣어 계산하고 21개로 나눈다 (`src.general probs`).
`probs.jsonl` 형식이 같아서 `refusal_metrics`·`negcat_report`·`style_report` 를 그대로 쓴다.
척도가 달라 **ours 와 ND·G 의 문턱값 숫자는 직접 비교하지 않는다** (AUROC·거절 개수로 비교. ND vs G 는 척도가 같다).

**런타임**: Colab **A100 80GB + 고용량 RAM** (ours 와 같은 장비. 40GB 가 배정되면 준비 셀에서 멈춘다). Colab Secrets 에 `HF_TOKEN` 필요. G 는 문장이 ours 의 약 21배라 학습이 오래 걸린다 (2번 시험 학습으로 먼저 잰다).

## 준비 (그냥 실행)

In [ ]:
# 다른 노트북과 같은 버전 (최신 5.x 는 API 가 바뀌어 에러가 난다)
!pip -q install transformers==4.56.2 accelerate
from huggingface_hub import login
from google.colab import userdata, drive

login(userdata.get("HF_TOKEN"))
drive.mount("/content/drive")

import os
REPO = "/content/octopus_v2_reproduce"
if not os.path.exists(REPO):
    !git clone https://github.com/nahcub/octopus_v2_reproduce.git {REPO}
%cd {REPO}
!git fetch -q origin
!git checkout exp/general-json   # main 에 합쳐지기 전까지 이 브랜치
!git pull
!git log --oneline -1

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import transformers, os, json, datetime; print("transformers", transformers.__version__)  # 4.56.2 여야 함

DRIVE = "/content/drive/MyDrive/octopus_v2"
OURS_DIR = f"{DRIVE}/runs/full"
OFFICIAL = "data/raw/official/official_test.jsonl"

# 학습 옵션. 나머지(lr, epoch, warm-up, seed 0, optimizer)는 train.py 기본값 = ours 와 같음.
# 실제 batch = batch-size × grad-accum = 16. ND 는 ours 와 길이가 비슷해 ours 처럼 한 번에 16 (OOM 이면 --batch-size 8 --grad-accum 2)
TRAIN_ARGS = {
    "ND": "--format json --no-list --json-desc",
    "G":  "--format json --batch-size 4 --grad-accum 4 --gradient-checkpointing",
}
def model_dir(name): return f"{DRIVE}/runs/{name}"

# GPU 확인: ours 와 같은 조건(fp32 가중치 + AdamW, batch 16)은 A100 80GB 에서만 들어간다 (노트북 10 과 같은 확인).
# 40GB·48GB 에서는 옵티마이저 갱신 단계에서 OOM (2026-11-25 A6000 48GB 에서 확인). 8-bit Adam 등으로 바꾸면 ours 와 조건이 달라진다
import torch
GPU_GB = torch.cuda.get_device_properties(0).total_memory / 2**30
assert GPU_GB > 60, f"GPU {GPU_GB:.0f} GiB: A100 80GB 가 필요하다. 런타임 연결 해제 후 A100 + 고용량 RAM 으로 다시 연결"
print(f"GPU {GPU_GB:.0f} GiB OK")

# 이름: (test 파일, 결과 폴더, ours 의 기존 결과 폴더)
SETS = {
    "dev":      ("data/test.jsonl",               f"{DRIVE}/runs/eval",          f"{DRIVE}/runs/eval/ours"),
    "official": (OFFICIAL,                        f"{DRIVE}/runs/eval_official", f"{DRIVE}/runs/eval_official/ours"),
    "style":    ("data/style/dev_styled.jsonl",   f"{DRIVE}/runs/eval/style",    f"{DRIVE}/runs/eval/style/ours"),
}
def n_lines(p): return sum(1 for _ in open(p)) if os.path.exists(p) else -1
def done_eval(name, s):
    test, base, _ = SETS[s]
    return n_lines(f"{base}/{name}/probs.jsonl") == n_lines(test) and n_lines(f"{base}/{name}/preds.jsonl") == n_lines(test)
if not os.path.exists(OFFICIAL):
    !python -m src.official_bench convert
for name in TRAIN_ARGS:
    print(f"{name:<3} 학습 끝: {os.path.exists(model_dir(name) + '/train_args.json')} |",
          " ".join(f"{s} 채점 끝: {done_eval(name, s)}" for s in SETS))
for s in SETS: print(s, "| ours 기존 probs:", os.path.exists(f"{SETS[s][2]}/probs.jsonl"))

---
## 1. 데이터 확인
프롬프트를 눈으로 보고 길이를 잰다 (G: 목록 있음, ND: 목록 없음 + 정답 뒤 설명). 정답을 기존 채점기 형식으로 바꾼 것이 원래 정답과 같은지도 본다.

**예측**: ND·G 는 우리 test 에서 ours 보다 positive 정확도가 높을까 낮을까?
> (여기에 예측)

In [ ]:
!python -m src.general show
!python -m src.general stats

## 2. 시험 학습 (30 step) — 메모리·속도 재기
확인할 것: `[데이터]` 4000개·최대 토큰 수 (ND 약 374, G 약 2,500), `[학습 전 loss]`, 최대 메모리, step 당 시간 → 본 학습 시간 = step 당 시간 × 750.
메모리가 넘치면 `--batch-size` 를 반으로, `--grad-accum` 을 두 배로 (실제 batch 16 유지). 결과를 보고 준비 셀의 `TRAIN_ARGS` 를 고친다.

In [ ]:
for name in TRAIN_ARGS:
    print(f"\n==================== {name}")
    !python -m src.train {TRAIN_ARGS[name]} --max-steps 30 --output-dir /content/runs/try_{name}

---
## 3. 본 학습 → 생성·채점·거절 점수 (dev, 공식 210, 말투 test)
이미 끝난 것은 건너뛴다. 생성은 `src.general generate` (JSON 뒤 빈 줄에서 멈춤, 목록을 넣을지는 모델 폴더의 `train_args.json` 을 읽어 정함),
채점은 기존 `src.evaluate score` 그대로, 거절 점수는 `src.general probs`.

In [ ]:
for name in TRAIN_ARGS:
    mdir = model_dir(name)
    if not os.path.exists(f"{mdir}/train_args.json"):
        print(f"\n#################### {name} 학습 시작", datetime.datetime.now())
        !python -m src.train {TRAIN_ARGS[name]} --output-dir {mdir}
        print(f"#################### {name} 학습 끝", datetime.datetime.now())
    for s, (test, base, _) in SETS.items():
        if done_eval(name, s):
            continue
        out = f"{base}/{name}"
        print(f"\n==================== {name} / {s}", datetime.datetime.now())
        !python -m src.general generate --model {mdir} --test-file {test} --out {out}/preds.jsonl
        !python -m src.evaluate score --test-file {test} --preds {out}/preds.jsonl --out {out}
        !python -m src.general probs --model {mdir} --test-file {test} --preds {out}/preds.jsonl --out {out}/probs.jsonl

---
## 4. 비교 — 핵심 표
1. 정답 채점 (greedy): 전체 정확도, positive / negative. ND·G 가 ours 와 크게 다르면 학습이 덜 됐을 수 있으니 해석할 때 같이 적는다
2. 거절 지표: AUROC, dev 문턱값을 공식 210 에 옮겼을 때 positive 거절 수 (과잉 거절), 95% 구간 (짝지은 bootstrap). **ours vs G** 가 주 비교, 다르면 ours vs ND / ND vs G 로 원인
3. negative 범주별 (near-miss 인 smart_home·phone_settings·other_apps 에서 오호출이 다른가)
4. 말투별 (실험 S 와 같은 표. 각 모델도 말투에 따라 거절이 흔들리나)

**예측**: 공식 210 의 낯선 말투 positive 과잉 거절(ours 65개)은 G 에서 줄어들까? 줄어든다면 ND 에서도 줄어들까?
> (여기에 예측)

In [ ]:
NAMES = ["ours", "ND", "G"]
def res_dir(name, s): return SETS[s][2] if name == "ours" else f"{SETS[s][1]}/{name}"
for s in ("dev", "official"):
    print(f"\n==================== {s} (greedy 채점)")
    for name in NAMES:
        d = res_dir(name, s)
        if os.path.exists(f"{d}/summary.json"):
            m = json.load(open(f"{d}/summary.json"))["전체"]
            print(f"  {name:<5} 전체 {m['acc']:.2%} | positive {m['acc_positive']:.2%} | negative {m['acc_negative']:.2%} | {m['outcomes']}")

In [ ]:
def models_arg(s, dev_thresh):
    # --model 이름 probs [문턱값을 잡을 dev probs 또는 -]
    return " ".join(f"--model {n} {res_dir(n, s)}/probs.jsonl {res_dir(n, 'dev') + '/probs.jsonl' if dev_thresh else '-'}" for n in NAMES)
D, O = SETS["dev"], SETS["official"]
print("== 우리 test (dev) 거절 지표")
!python -m src.refusal_metrics --test {D[0]} {models_arg("dev", False)}
print("== 공식 210 (문턱값은 각 모델의 dev 에서)")
!python -m src.refusal_metrics --test {O[0]} {models_arg("official", True)}
print("== negative 범주별 (dev)")
cats = " ".join(f"--model {n}:irr {res_dir(n, 'dev')}/probs.jsonl" for n in NAMES)
!python -m src.negcat_report --test-file {D[0]} {cats}

In [ ]:
S = SETS["style"]
st = " ".join(f"--model {n} {res_dir(n, 'style')}/probs.jsonl" for n in NAMES)
os.makedirs(f"{S[1]}/G_ND", exist_ok=True)
!python -m src.style_report --test-file {S[0]} {st} --out {S[1]}/G_ND/style_report.json | tee {S[1]}/G_ND/style_report.txt

---
## 끝나면
결과: Drive `runs/{ND,G}/` (모델), `runs/eval/{ND,G}/`, `runs/eval_official/{ND,G}/`, `runs/eval/style/{ND,G}/` (`preds`·`scored`·`summary`·`probs`), 말투 표 `runs/eval/style/G_ND/`.
랩탑에서 Claude 에게 "실험 G·ND 결과 분석해줘" 라고 하면 설계 문서 4.5절 예측·판정 규칙과 비교한다.
MASSIVE·BFCL 채점은 모든 모델이 확정된 뒤 한 번에 한다 (설계 문서 8절). ND·G 도 그때 같이.